# Lab 2, Part 1: from S4 to S6 — what selection adds to a pretrained Mamba

Checkpoints (each gates the next stage):
- **CP1** baseline English ppl ≈ 10.56
- **CP2** V4 == stock logits (max abs diff < 1e-3, fp32, eval)
- **CP3** train-mode == eval-mode logits for every variant
- **CP4** kernel call counter: fused path in training, selective-scan in eval, no fallback
- **CP5** V4 after recovery not worse than baseline

Runs on one H100 (allowed for Part 1 only). Same numeric recipe as the T4 plan: fp32 weights, fp16 autocast + GradScaler.

## 0. Environment (Route B: mamba-ssm + causal-conv1d prebuilt wheels, torch ≤ 2.10)

In [ ]:
import torch, sys
print(sys.version, torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))
!nvcc --version | tail -2

In [ ]:
# Route B. If the wheels do not match this torch, use Route A: pip install kernels, then the Hub kernels
# kernels-community/mamba-ssm with an explicit version (check kernels.has_kernel) and Mode.TRAINING.
!MAMBA_KEEP_CUDA_BUILD=TRUE pip install -q mamba-ssm==2.3.2.post1 --no-build-isolation
!pip install -q causal-conv1d==1.7.0 --no-build-isolation
!pip install -q -e ..

In [ ]:
import importlib.metadata as im, json, os
from pathlib import Path
import pandas as pd
from transformers import AutoTokenizer, MambaForCausalLM
from lab2.common import load_config, out_dir
from lab2 import wandb_log as wl
from lab2.part1.variants import KERNELS, VARIANTS, apply_variant, Calibrator, VariantChecks
from lab2.part1.data import EvalSamples, TrainStream, overlap
from lab2.part1.evaluate import KernelCounter, LMEvaluator
from lab2.part1.recovery import RecoveryTrainer, param_counts, lr_factor

assert KERNELS, "mamba_ssm / causal_conv1d not importable: training would fall back to the slow path"
cfg = load_config()
p1, SEED, DEV = cfg["part1"], cfg["seed"], "cuda"
OUT = out_dir(cfg, "part1")                 # large outputs (samples, stream, checkpoints)
FIG = Path("../report/figures/part1")       # small tables / figures for the report
FIG.mkdir(parents=True, exist_ok=True)
ENV = {"torch": torch.__version__, "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0),
       **{k: im.version(k) for k in ("transformers", "mamba_ssm", "causal_conv1d")}}
print(ENV)
# os.environ["WANDB_API_KEY"] = "..."   # W&B is disabled without a key; everything is also saved locally

RESULTS = OUT / "results.json"
results = json.loads(RESULTS.read_text()) if RESULTS.exists() else {}
def save_result(key, value):
    results[key] = value
    RESULTS.write_text(json.dumps(results, indent=1))

tok = AutoTokenizer.from_pretrained(p1["model"])
def load_stock():
    return MambaForCausalLM.from_pretrained(p1["model"], dtype=torch.float32).to(DEV)

## 1. Fixed evaluation samples and the training stream (built once, then only loaded)

In [ ]:
samples_path = OUT / "eval_samples.pt"
es = EvalSamples(tok, p1["seq_len"])
if not samples_path.exists():
    torch.save(es.build_all(p1["data"], SEED), samples_path)
samples = torch.load(samples_path)
for name, s in samples.items():
    print(name, "docs", len(s["ids"]), "tokens", s["tokens"], "bytes", int(s["bytes"].sum()))
assert len(samples["en"]["ids"]) >= p1["data"]["min_docs"] and len(samples["uk"]["ids"]) >= p1["data"]["min_docs"]

# reproducibility: rebuild the (small) val sample with the same seed -> identical ids
pile = f"hf://datasets/{p1['data']['pile']}"
from lab2.part1.data import stream_texts
val_again = es.build(stream_texts(f"{pile}/{p1['data']['val_file']}", SEED, p1["data"]["shuffle_buffer"]), p1["data"]["val_tokens"])
assert EvalSamples.same(samples["val"], val_again)
short = sum(len(x) < p1["seq_len"] for x in samples["en"]["ids"])
print("en docs shorter than seq_len (kept whole):", short)

In [ ]:
stream_dir = out_dir(cfg, "part1", "stream")
if not (stream_dir / "train.npy").exists():
    print(TrainStream(tok, p1["seq_len"]).build(p1["data"], SEED, stream_dir))
calib_rows, train_rows = TrainStream.load(stream_dir)
print("calib rows", tuple(calib_rows.shape), "train rows", tuple(train_rows.shape))
ov = overlap((stream_dir / "train_keys.txt").read_text().split(), samples)
print("eval docs found in calib/train stream:", ov)
assert not any(ov.values())

## 2. Baseline (V4 zero-shot = unmodified model) — CP1, CP4

In [ ]:
wl.init_run(cfg, "baseline", "eval", {**p1, "env": ENV, "seed": SEED})
model = load_stock()
# CP4 on the training path: one fp16-autocast step must hit the fused kernel
counter = KernelCounter()
with counter():
    model.train()
    with torch.autocast("cuda", dtype=torch.float16):
        model(input_ids=train_rows[:2].to(DEV), labels=train_rows[:2].to(DEV), use_cache=False).loss.backward()
print("train step:", counter.assert_path("fused"))
model.zero_grad(set_to_none=True)

ev = LMEvaluator(DEV)
base = {name: ev.evaluate(model, samples[name]) for name in ("en", "uk")}
print(json.dumps(base, indent=1))
save_result("zeroshot/V4", base)
wl.log_summary({"en/ppl": base["en"]["ppl"], "en/bpb": base["en"]["bpb"], "uk/bpb": base["uk"]["bpb"],
                "published_ppl": p1["published_ppl"], **param_counts(model)})
wl.finish()
print(f"CP1: ppl {base['en']['ppl']:.3f} vs published {p1['published_ppl']} (test vs validation split)")

## 3. Calibration of β, B, C on ~1M train tokens (stock model, eval mode)

In [ ]:
calib_path = OUT / "calibration.pt"
wl.init_run(cfg, "calibration", "calibrate", {"tokens": int(calib_rows.numel())})
if not calib_path.exists():
    Calibrator.save(Calibrator(load_stock()).run(calib_rows), calib_path)
calib = Calibrator.load(calib_path)
ratios = pd.DataFrame({"layer": range(len(calib["ratio_B"])), "ratio_B": calib["ratio_B"].numpy(),
                       "ratio_C": calib["ratio_C"].numpy()})
wl.log_table("calibration_ratio", ratios, FIG / "calibration_ratio")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ratios.layer, ratios.ratio_B, marker="o", label="B")
ax.plot(ratios.layer, ratios.ratio_C, marker="o", label="C")
ax.set(xlabel="layer", ylabel="‖mean‖ / mean‖·‖", title="How much of B_k, C_k survives averaging")
ax.legend()
wl.log_figure("calibration_ratio_plot", fig, FIG / "calibration_ratio.png")
wl.finish()
ratios

## 4. Variant checks — CP2, CP3, static invariance, gradient reproducibility

In [ ]:
wl.init_run(cfg, "checks", "test")
build = lambda v: apply_variant(load_stock(), v, calib)
batch = train_rows[:2, :1024].to(DEV)                      # unpadded
checks = VariantChecks(build, load_stock(), batch, p1["checks"]["v4_tol"], p1["checks"]["grad_rtol"])
counter = KernelCounter()
with counter():
    check_rows = checks.run_checks()
checks_df = pd.DataFrame(check_rows)
wl.log_table("checks", checks_df, FIG / "checks")
wl.log_summary({"kernel_counts": dict(counter.counts)})
wl.finish()
checks_df

## 5. Zero-shot surgery

**Pre-registration:** write the predicted V1–V4 ordering by zero-shot English perplexity into
`report/part1.md` and commit it *before* running the next cell.

In [ ]:
for v in ("V1", "V2", "V3"):
    if f"zeroshot/{v}" in results:
        continue
    wl.init_run(cfg, f"zeroshot-{v}", "eval", {"variant": v})
    model = build(v)
    r = {name: ev.evaluate(model, samples[name]) for name in ("en", "uk")}
    save_result(f"zeroshot/{v}", r)
    wl.log_summary({"en/ppl": r["en"]["ppl"], "en/bpb": r["en"]["bpb"], "uk/bpb": r["uk"]["bpb"], **param_counts(model)})
    wl.finish()
pd.DataFrame({v: {"en_ppl": results[f"zeroshot/{v}"]["en"]["ppl"], "table7": p1["table7_ppl"][v]} for v in VARIANTS}).T

## 6. Recovery training (same rows, order, budget, seed, hyperparameters) — CP4, CP5

In [ ]:
trainer = RecoveryTrainer(cfg, ev, samples["val"])

def recover(v, seed, max_steps=None):
    key = f"recovery/{v}/s{seed}"
    if key in results and max_steps is None:
        return results[key]
    run_dir = out_dir(cfg, "part1", "runs", f"{v}-s{seed}")
    wl.init_run(cfg, f"recovery-{v}-s{seed}", "train", {**p1["train"], "variant": v, "seed": seed, "env": ENV},
                group=f"recovery-s{seed}")
    model = build(v)
    history, summary = trainer.train(model, train_rows, seed, run_dir, max_steps)
    r = {name: ev.evaluate(model, samples[name]) for name in ("en", "uk")}
    wl.log_summary({"en/ppl": r["en"]["ppl"], "en/bpb": r["en"]["bpb"], "uk/bpb": r["uk"]["bpb"]})
    wl.finish()
    out = {**r, "history": history, "summary": summary}
    if max_steps is None:
        save_result(key, out)
    return out

# timing / sanity: ~20 steps of V4 (fit micro_batch in config.yaml if it does not fit)
t = recover("V4", SEED, max_steps=20)
print(t["summary"])

In [ ]:
v4 = recover("V4", SEED)
print("CP5:", v4["en"]["ppl"], "<=", base["en"]["ppl"])
assert v4["en"]["ppl"] <= base["en"]["ppl"], "V4 got worse with recovery: lower the lr and redo before V1-V3"

In [ ]:
for v in ("V1", "V2", "V3"):
    recover(v, SEED)

In [ ]:
# second seed for the two variants closest in English ppl after recovery
ppl = {v: results[f"recovery/{v}/s{SEED}"]["en"]["ppl"] for v in VARIANTS}
ordered = sorted(ppl, key=ppl.get)
pair = min(zip(ordered, ordered[1:]), key=lambda ab: abs(ppl[ab[0]] - ppl[ab[1]]))
seed2 = p1["train"]["seeds"][1]
print("closest pair:", pair, "-> seed", seed2)
for v in pair:
    recover(v, seed2)

## 7. Report tables and figures

In [ ]:
rows = []
for v in VARIANTS:
    z, r = results[f"zeroshot/{v}"], results[f"recovery/{v}/s{SEED}"]
    row = {"variant": v, "table7_ppl": p1["table7_ppl"][v],
           "zs_en_ppl": z["en"]["ppl"], "zs_en_bpb": z["en"]["bpb"], "zs_uk_bpb": z["uk"]["bpb"],
           "rec_en_ppl": r["en"]["ppl"], "rec_en_bpb": r["en"]["bpb"], "rec_uk_bpb": r["uk"]["bpb"],
           "params_trainable": r["summary"]["params_trainable"], "train_hours": r["summary"]["wall_seconds"] / 3600,
           "step_seconds": r["summary"]["step_seconds"]}
    s2 = results.get(f"recovery/{v}/s{seed2}")
    if s2:
        row.update({"rec_en_ppl_s2": s2["en"]["ppl"], "rec_uk_bpb_s2": s2["uk"]["bpb"]})
    rows.append(row)
table = pd.DataFrame(rows)
table.loc[table.variant == "V4", "published_ppl"] = p1["published_ppl"]
wl.init_run(cfg, "report", "report")
wl.log_table("results", table, FIG / "results")

fig, ax = plt.subplots(figsize=(8, 4.5))
for v in VARIANTS:
    h = results[f"recovery/{v}/s{SEED}"]["history"]
    ax.plot([p["tokens"] / 1e6 for p in h], [p["val_loss"] for p in h], marker=".", label=v)
ax.set(xlabel="training tokens (M)", ylabel="val loss (200k-token sample)", title="Recovery curves", yscale="log")
ax.legend()
wl.log_figure("recovery_curves", fig, FIG / "recovery_curves.png")
kernels = pd.DataFrame([{"run": k, **v["summary"]["kernel_counts"]} for k, v in results.items() if k.startswith("recovery")])
wl.log_table("kernel_counts", kernels, FIG / "kernel_counts")
wl.finish()
table